# Demo on Colab — mirrors the production mission flow

This notebook walks through the exact same sequence the real robot will run, just with photo uploads standing in for hardware.

**Production flow:**

| Phase | What happens in production | What you do here |
|---|---|---|
| 1. Report | A reporter snaps a tight crop of the trash + a wider context shot, uploads via Expo app to the relay. | Upload your own REF + CTX photos (or use the bundled test pair). |
| 2. Navigate | Robot's mounted phone drives the robot via Apple Maps waypoints + GPS to the report's location. | Skipped — A100 has no GPS / motors. Assume we've arrived. |
| 3. Approach | At the last waypoint, the brain takes over the WebSocket to the Pi. Each camera frame → OWLv2 (find target) → if hidden, Qwen3-VL scout (which way to rotate) → discrete Action → motor PWM. | Upload live-style frames one at a time. The brain processes each, shows the Action. Controller state (search bursts, etc.) carries across frames. |
| 4. Verify / report | Robot confirms target gone (driven over with scoop). POST result to relay. | Trajectory cell shows whether STOP was reached, mirroring the success signal. |

**Setup:** Runtime → Change runtime type → **A100 GPU**, then run cells top to bottom.

## 0. Set up the Colab runtime (once per session)

Clones the repo + installs HuggingFace transformers. ~2 min.

In [ ]:
!git clone https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git
%cd RU-IEEE-Hackathon
!pip install -q -U git+https://github.com/huggingface/transformers@main accelerate bitsandbytes pillow

## 1. Load the brain (once per session)

Loads OWLv2 + Qwen3-VL-8B at fp16. ~3-5 min. After this, every step is fast.

**Production analog:** This is the brain process starting up on the RTX 4080. In production it boots once at the start of the day and stays loaded for hours.

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

REPO = Path.cwd()
sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'no GPU — Runtime → Change runtime type → A100'
print(f'GPU: {torch.cuda.get_device_name(0)}  ·  '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')

from brain.perception.target_finder import TargetFinder
from brain.perception.vlm_scout import VLMScout
from brain.control.loop import Action, ApproachController
from brain.control.action_to_pwm import pwm_for

print('loading OWLv2 (~300 MB)...')
finder = TargetFinder(model_name='google/owlv2-base-patch16-ensemble', min_sim=0.3)

print('loading Qwen3-VL-8B at fp16 (~18 GB on GPU)...')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=False)
print(f'  loaded in {time.monotonic() - t0:.1f}s  ·  '
      f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('\nbrain ready. No mission loaded yet — go to the next cell.')

## 2. Receive a trash report (the reporter's submission)

**Production analog:** A reporter takes two phone photos — a tight crop of the trash + a wider scene with landmarks — and POSTs them through the Expo app to the relay (`POST /reports`).

Here, choose one path:
- **Quick test:** use the bundled `references/ref.jpg` + `references/ctx.jpg` already in the repo. Skip the upload cell, run the next one.
- **Real test:** upload your own pair via the upload cell — match the production scenario more closely.

In [ ]:
# Real test: upload your own REF (tight bottle crop) and CTX (wider shot with landmarks).
# Skip this cell to use the bundled references in the repo.
from google.colab import files
print('upload TWO files: ref (tight crop) and ctx (wider scene). Order doesn\'t matter.')
uploaded = files.upload()
names = list(uploaded.keys())
assert len(names) == 2, 'upload exactly 2 files'
# Heuristic: smaller file = tighter crop = REF. Larger = wider = CTX.
names_by_size = sorted(names, key=lambda n: len(uploaded[n]))
ref_name, ctx_name = names_by_size[0], names_by_size[1]
print(f'using {ref_name} as REF (tight crop)')
print(f'using {ctx_name} as CTX (wider scene)')
REF_OVERRIDE = Path(ref_name)
CTX_OVERRIDE = Path(ctx_name)

In [ ]:
# Pick which references to use. Falls back to bundled if nothing was uploaded above.
REF = REF_OVERRIDE if 'REF_OVERRIDE' in dir() else REPO / 'references' / 'ref.jpg'
CTX = CTX_OVERRIDE if 'CTX_OVERRIDE' in dir() else REPO / 'references' / 'ctx.jpg'
assert REF.exists() and CTX.exists(), f'missing {REF} or {CTX}'

ref_bgr = cv2.imread(str(REF))
ctx_bgr = cv2.imread(str(CTX))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'REF (tight crop of target)\n{REF.name}')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(ctx_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'CTX (wider scene with landmarks)\n{CTX.name}')
axes[1].axis('off')
plt.show()

print('\nCheck: does REF show ≥70% target on a plain background?')
print('       does CTX show the same target plus landmarks (bench, floor, chair)?')
print('If not, recapture and re-run the upload cell above.')

## 3. Mission start — bind references into the brain

**Production analog:** When the brain accepts a new job, it loads the reporter's REF into OWLv2 (encodes once, caches the query embedding) and creates a fresh ApproachController bound to both REF and CTX. This cell does both.

In [ ]:
finder.load_reference(ref_bgr)
controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)
history = []
print('mission armed. Brain is now hunting for THIS specific target.')
print('Next: feed it live frames.')

## 4. Approach loop — feed live frames, get Actions

**Production analog:** The brain reads frames from the Pi's MJPEG stream at ~15 FPS. For each frame, `controller.step(frame)` returns one Action. That action gets translated to PWM via `action_to_pwm.py` and sent to the Pi over WebSocket. The Pi's L298N drives the motors.

Here, you upload "live" frames one at a time. The controller's internal state — including the 15-frame SEARCH burst counter that prevents calling Qwen3-VL on every frame — **persists across uploads**, exactly as it would across real camera frames.

**Tip:** take 5-10 phone photos of the bottle scene at varying poses BEFORE starting:
- bottle hidden / out of frame
- bottle at the edge of frame
- bottle centered far away
- bottle centered close up
- bottle filling the frame (close enough to STOP)

Upload them sequentially — that's the closest analog to a real camera stream.

In [ ]:
MIN_SIM = 0.3   # raise to 0.6+ if OWLv2 saturates with false positives
finder.min_sim = MIN_SIM

ACTION_LABEL = {
    Action.FORWARD:      'WALK FORWARD',
    Action.LEFT:         'TURN LEFT',
    Action.RIGHT:        'TURN RIGHT',
    Action.STOP:         'STOP — TARGET REACHED',
    Action.SEARCH_LEFT:  'ROTATE LEFT (scanning)',
    Action.SEARCH_RIGHT: 'ROTATE RIGHT (scanning)',
}
ACTION_COLOR = {
    Action.FORWARD: '#00cc00', Action.LEFT: '#ff8800',
    Action.RIGHT: '#ff8800', Action.STOP: '#cc0000',
    Action.SEARCH_LEFT: '#cc00cc', Action.SEARCH_RIGHT: '#cc00cc',
}
ACTION_ARROW = {
    Action.FORWARD: '\u2191', Action.LEFT: '\u2190',
    Action.RIGHT: '\u2192', Action.STOP: '\u25A0',
    Action.SEARCH_LEFT: '\u21BA', Action.SEARCH_RIGHT: '\u21BB',
}

def process_frame(frame, source_label=''):
    """Run the brain on one BGR frame, display the Action. Same code path the
    real robot uses per camera frame: detect → controller.step → pwm_for."""
    t0 = time.monotonic()
    detections = finder.detect(frame)
    action = controller.step(frame)
    elapsed_ms = (time.monotonic() - t0) * 1000
    left, right = pwm_for(action)
    history.append((action, len(detections), elapsed_ms, source_label))

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.05)
    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if detections:
        d = detections[0]
        x1, y1, x2, y2 = d.xyxy
        ax_img.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor='lime', linewidth=4))
        ax_img.text(x1, max(y1-8, 16), f'target  {d.confidence:.2f}',
                    color='black', fontsize=14, weight='bold',
                    bbox=dict(facecolor='lime', alpha=0.85, pad=4))
    ax_img.set_title(source_label, fontsize=11, color='#666', loc='left')
    ax_img.axis('off')

    ax_act = fig.add_subplot(gs[1])
    ax_act.set_xlim(0, 1); ax_act.set_ylim(0, 1); ax_act.axis('off')
    color = ACTION_COLOR[action]
    ax_act.text(0.5, 0.65,
                f'{ACTION_ARROW[action]}  {ACTION_LABEL[action]}  {ACTION_ARROW[action]}',
                ha='center', va='center', fontsize=44, color=color, weight='bold')
    ax_act.text(0.5, 0.2,
                f'frame #{len(history)}   ·   {len(detections)} detection(s)   ·   '
                f'pwm = ({left:+d}, {right:+d})   ·   {elapsed_ms:.0f} ms',
                ha='center', va='center', fontsize=13, color='#666')
    plt.show()
    return action

def upload_and_process():
    """Single-frame upload — closest analog to one camera frame arriving from the Pi."""
    print('upload one "live" frame from the robot\'s view:')
    uploaded = files.upload()
    if not uploaded:
        print('no file uploaded')
        return None
    name = next(iter(uploaded.keys()))
    frame = cv2.imread(name)
    assert frame is not None, f'could not read {name}'
    return process_frame(frame, source_label=name)

_ = upload_and_process()

**Re-run the cell above** to feed the next frame. The controller's internal state carries over — so if a frame triggered a SEARCH burst, the next ~14 uploads will continue rotating in that direction even if OWLv2 still sees nothing.

Or, batch-process a sequence of frames at once (closer to a real video stream):

In [ ]:
# Batch — upload an ORDERED sequence of frames (Ctrl/Cmd-click to select multiple).
# Treat each upload as one camera tick. Goal: see the Action change as the bottle
# enters frame, gets centered, fills frame, etc.
print('select a sequence of frames (ordered as if streamed from the Pi):')
uploaded = files.upload()
for name in sorted(uploaded):  # process in name order — name your files frame_001.jpg, frame_002.jpg, ...
    frame = cv2.imread(name)
    if frame is None:
        print(f'{name}: could not read — skipping')
        continue
    process_frame(frame, source_label=name)

## 5. Verify / report — did we reach the target?

**Production analog:** When the controller emits STOP and the next frame's OWLv2 detection drops to zero (the bottle is now under the scoop, not visible to the camera), the FSM transitions to VERIFYING → REPORTING (success), POSTs the result to the relay. The reporter's submission is marked complete in the database.

Here, the trajectory log shows whether the brain converged to a stable STOP.

In [ ]:
if not history:
    print('no frames processed yet')
else:
    print(f'{"#":>3}  {"action":<14}  {"dets":>4}  {"step ms":>8}  source')
    print('-' * 60)
    for i, (act, n, ms, src) in enumerate(history, 1):
        print(f'{i:>3}  {act.name:<14}  {n:>4}  {ms:>8.0f}  {src}')

    last5 = [a.name for a, _, _, _ in history[-5:]]
    print()
    if len(last5) == 5 and len(set(last5)) > 3:
        print('⚠️  controller is oscillating — last 5 actions are noisy. ALIGN_TOLERANCE may need tuning.')
    elif last5 == ['STOP'] * 5:
        print('✅ stable STOP — mission would now transition to VERIFYING → REPORTING (success).')
    elif Action.STOP.name in last5:
        print('🟡 STOP appeared but not yet stable — the next frame should confirm.')
    else:
        actions_seen = set(a.name for a, _, _, _ in history)
        print(f'mission in progress — actions seen: {actions_seen}')

## 6. New mission (reset)

**Production analog:** After REPORTING, the FSM returns to IDLE and polls the relay for the next pending report. Each new report = new REF/CTX = fresh controller state.

Run this cell to clear history + reset controller state, then go back to Cell 2 (upload new REF/CTX) for the next "mission."

In [ ]:
history.clear()
controller._search_remaining = 0
controller._search_direction = None
# Drop the override so the next mission re-prompts.
for var in ('REF_OVERRIDE', 'CTX_OVERRIDE'):
    globals().pop(var, None)
print('mission reset. Go to Cell 2 to upload a new report.')

## What this proves about the production system

If you run a full mission (upload REF/CTX → upload a sequence of live frames → see Actions converge to STOP), you've validated:
- The brain accepts arbitrary reporter photos and binds them as the mission target (OWLv2 query embedding cached, controller wired to both refs).
- The same code path the real robot uses (`detect → controller.step → pwm_for`) runs end-to-end on real images.
- The controller's stateful logic (search bursts, STOP detection) works across a stream of frames.
- Qwen3-VL produces usable directional guidance from the three-image prompt.

What's left for production isn't more brain code — it's wiring the brain to the Pi (already implemented in `brain/io/pi_bridge.py`), the iPhone GPS listener (already implemented in `brain/io/iphone_listener.py`), the relay backend (Node, not yet built), and the phone-side nav loop (your teammate's `robot-console/`).